# Supervised Fine-Tuning (SFT) with alignrl

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sacredvoid/alignrl/blob/main/notebooks/01_sft_instruction_tuning.ipynb)

This notebook walks through **instruction-tuning a language model** using Supervised Fine-Tuning (SFT) with QLoRA. SFT is typically the first post-training step: you take a pretrained base model and teach it to follow instructions by training on (instruction, response) pairs. We use QLoRA (4-bit quantized LoRA) to make this feasible on a free Colab T4 GPU.

## Setup

Install alignrl with training and Unsloth dependencies. This takes 2-3 minutes on Colab.

In [ ]:
!pip install alignrl[train,unsloth] -q

## How SFT + QLoRA Works

**Supervised Fine-Tuning** is straightforward: given a dataset of high-quality instruction-response pairs, we minimize the cross-entropy loss on the response tokens. The model learns to generate helpful, well-structured answers.

**QLoRA** makes this memory-efficient:
1. The base model weights are quantized to 4-bit (NormalFloat4)
2. Small trainable LoRA adapters (rank 16) are inserted into attention and MLP layers
3. Only the adapter weights are updated during training

This reduces GPU memory from ~24GB (full fine-tuning) to ~6GB (QLoRA), making it possible to fine-tune a 3B parameter model on a T4.

```
Base Model (frozen, 4-bit)     LoRA Adapters (trainable, fp16)
+-----------------------+      +--------+
|  Qwen2.5-3B weights   | ---> | r=16   | ---> Output
|  (quantized)          |      | alpha=32|
+-----------------------+      +--------+
     ~2GB VRAM                   ~50MB VRAM
```

## Configuration

We create an `SFTConfig` that specifies the model, dataset, LoRA parameters, and training hyperparameters. Key choices:
- **dataset_size=1000**: Use a small subset of OpenHermes-2.5 so training finishes in ~10 minutes on Colab
- **max_steps=100**: Cap training to 100 gradient steps
- **report_to="none"**: Disable W&B logging for this notebook

In [ ]:
from alignrl.sft import SFTConfig, SFTRunner

config = SFTConfig(
    model_name="Qwen/Qwen2.5-3B",
    output_dir="./outputs/sft",
    dataset_name="teknium/OpenHermes-2.5",
    dataset_size=1000,
    max_steps=100,
    max_seq_length=2048,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lora_r=16,
    lora_alpha=32,
    load_in_4bit=True,
    report_to="none",
    logging_steps=10,
)

print(config.model_dump_json(indent=2))

## Data Exploration

Let's look at the OpenHermes-2.5 dataset before training. Each example has a list of conversation turns with `from` (human/gpt/system) and `value` fields.

In [ ]:
from datasets import load_dataset

ds = load_dataset("teknium/OpenHermes-2.5", split="train")
print(f"Full dataset size: {len(ds):,} examples")
print(f"Columns: {ds.column_names}")
print()

# Look at a single example
example = ds[0]
print("=" * 60)
print("Example conversation:")
print("=" * 60)
for turn in example["conversations"]:
    role = turn["from"].upper()
    value = turn["value"][:200] + ("..." if len(turn["value"]) > 200 else "")
    print(f"\n[{role}]\n{value}")

Let's also look at how `alignrl` formats these conversations for training using the chat template:

In [ ]:
from alignrl.sft import format_instruction

messages = format_instruction(ds[0])
print("Formatted as chat messages:")
for msg in messages:
    content_preview = msg['content'][:100] + ("..." if len(msg['content']) > 100 else "")
    print(f"  {msg['role']}: {content_preview}")

## Before Training: Base Model Generation

Let's see how the base model (without any fine-tuning) responds to an instruction. We'll compare this with the fine-tuned model later.

In [ ]:
from alignrl.inference import InferenceConfig, ModelServer, build_prompt

# Load base model for comparison
base_server = ModelServer(InferenceConfig(
    model_name="Qwen/Qwen2.5-3B",
    backend="unsloth",
    max_tokens=256,
))
base_server.load()

test_prompt = "Explain the difference between a list and a tuple in Python."
messages = build_prompt(test_prompt, system="You are a helpful coding assistant.")
base_response = base_server.generate(messages)

print("BASE MODEL RESPONSE:")
print("-" * 40)
print(base_response)

## Training

Now we run SFT. The `SFTRunner` handles everything: loading the model with QLoRA, formatting the dataset, and running the TRL `SFTTrainer`.

In [ ]:
runner = SFTRunner(config)
result = runner.train()

print(f"Training complete!")
print(f"  Steps: {result.num_steps}")
print(f"  Final loss: {result.loss_history[-1]:.4f}")
print(f"  Adapter saved to: {result.output_dir}")

## Results: Training Loss Curve

A decreasing loss curve confirms the model is learning from the instruction data.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(result.loss_history, linewidth=2, color="#2563eb")
plt.xlabel("Logging Step")
plt.ylabel("Training Loss")
plt.title("SFT Training Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Loss went from {result.loss_history[0]:.4f} to {result.loss_history[-1]:.4f}")

## Inference: After Fine-Tuning

Let's load the fine-tuned adapter and compare the response to the same prompt.

In [ ]:
# Load fine-tuned model
ft_server = ModelServer(InferenceConfig(
    model_name="Qwen/Qwen2.5-3B",
    adapter_path="./outputs/sft/final",
    backend="unsloth",
    max_tokens=256,
))
ft_server.load()

ft_response = ft_server.generate(messages)

print("=" * 60)
print("BEFORE SFT (Base Model):")
print("=" * 60)
print(base_response)
print()
print("=" * 60)
print("AFTER SFT (Fine-Tuned):")
print("=" * 60)
print(ft_response)

Let's try a few more prompts to see the difference:

In [ ]:
test_prompts = [
    "Write a haiku about machine learning.",
    "What are the three laws of thermodynamics? Explain briefly.",
    "How do I reverse a linked list in Python?",
]

for prompt in test_prompts:
    msgs = build_prompt(prompt, system="You are a helpful assistant.")
    response = ft_server.generate(msgs)
    print(f"Q: {prompt}")
    print(f"A: {response[:300]}")
    print("-" * 60)

## Save Adapter Weights

The adapter is already saved during training. Let's verify the output files and save the training results.

In [ ]:
import json
from pathlib import Path

output_dir = Path("./outputs/sft")

# List saved files
print("Saved files:")
for f in sorted(output_dir.rglob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / 1024 / 1024
        print(f"  {f.relative_to(output_dir)} ({size_mb:.1f} MB)")

# Load and display training results
with open(output_dir / "train_result.json") as f:
    saved_results = json.load(f)

print(f"\nTraining metrics:")
print(json.dumps(saved_results["metrics"], indent=2))

## Next Steps

Now that we have an instruction-tuned model, the next stages in the alignrl pipeline are:

1. **GRPO** (Notebook 02) - Reinforce math reasoning with verifiable rewards
2. **DPO** (Notebook 03) - Align with human preferences
3. **Evaluation** (Notebook 04) - Benchmark across all stages
4. **Inference** (Notebook 05) - Serve and demo the trained model

The SFT adapter saved at `./outputs/sft/final` can be loaded in subsequent stages as the starting point.